# train-eval-mode-branch — worked example 2: Dropout variance collapses in eval mode

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `train-eval-mode-branch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

During training, `nn.Dropout(p)` randomly zeroes each element with probability `p`, introducing variance across repeated forward passes of the same input. In eval mode, Dropout is a no-op: the output is a scaled copy of the input (already compensated by the `1/(1-p)` training-time scale). Measuring variance across repeated passes reveals the mode switch clearly.

## Worked solution

**Step 1 – Build a Dropout module.** `nn.Dropout(p=0.4)` zeroes 40% of elements during training and is a no-op in eval mode.

**Step 2 – Repeated forward passes in train mode.** We call `model.train()` and run the same input `x` through the module `n_reps` times. Because each call draws a fresh mask, consecutive outputs differ. We compute the variance of the stacked outputs.

**Step 3 – Repeated forward passes in eval mode.** We call `model.eval()` and run the same input `n_reps` times. Every output is `x * 1.0` (the scaling compensation keeps the expected value correct). The variance across passes is exactly zero.

**Step 4 – Verify.** We check `var_train > 0` (stochastic) and `var_eval == 0.0` (deterministic). This is the simplest empirical proof that mode switching works.

In [ ]:
import torch as t
import torch.nn as nn

def worked2_dropout_variance_by_mode(p=0.4, n_reps=20):
    """
    In train mode, repeated Dropout passes have nonzero variance.
    In eval mode, variance is zero.
    Returns dict with variance values.
    """
    t.manual_seed(22)
    dropout = nn.Dropout(p=p)
    x = t.ones(10)  # simple input

    # Train mode: each pass drops different elements
    dropout.train()
    train_outs = t.stack([dropout(x) for _ in range(n_reps)])
    var_train = train_outs.var(dim=0).mean().item()

    # Eval mode: all passes produce identical output
    dropout.eval()
    eval_outs = t.stack([dropout(x) for _ in range(n_reps)])
    var_eval = eval_outs.var(dim=0).mean().item()

    return {
        'var_train': var_train,
        'var_eval': var_eval,
        'train_is_stochastic': var_train > 0,
        'eval_is_deterministic': var_eval == 0.0,
    }

result = worked2_dropout_variance_by_mode()
print('train variance:', f"{result['var_train']:.4f}")
print('eval variance:', result['var_eval'])
print('train stochastic:', result['train_is_stochastic'])
print('eval deterministic:', result['eval_is_deterministic'])